# Notebook 03 — Cohort Analysis

**Goal:** Group customers by their first-purchase month (cohort) and track how many return in subsequent months. Produce a retention heatmap and compute average retention rates at 1, 3, and 6 months.

**Business context:** Olist is a marketplace where customers often buy once for a specific need. Understanding whether cohorts return — and if not, why — is the first step toward designing retention programs.

**Pre-requisite:** Notebook 01 must have been run to create `sales_master.db`.

---

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

from src.db_setup import get_engine, DB_PATH
from src.analysis import (
    build_cohort_table,
    build_retention_rates,
    plot_cohort_heatmap,
    compute_avg_retention,
    save_fig,
)

sns.set_theme(style='whitegrid')
engine = get_engine()
print(f'Connected: {DB_PATH}')

## Step 1: Load Order + Customer Data

We need each order linked to its `customer_unique_id` — the consistent customer identifier across Olist's dataset (one person can have multiple `customer_id` values from different orders).

In [ ]:
orders_sql = '''
SELECT
    o.order_id,
    c.customer_unique_id,
    o.order_purchase_timestamp
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_purchase_timestamp IS NOT NULL
  AND o.order_status NOT IN ('canceled', 'unavailable')
'''
orders_df = pd.read_sql_query(orders_sql, engine)
orders_df['order_purchase_timestamp'] = pd.to_datetime(
    orders_df['order_purchase_timestamp'], errors='coerce'
)
orders_df = orders_df.dropna(subset=['order_purchase_timestamp'])

print(f'Orders loaded: {len(orders_df):,}')
print(f'Unique customers: {orders_df["customer_unique_id"].nunique():,}')
print(f'Date range: {orders_df["order_purchase_timestamp"].min().date()} → {orders_df["order_purchase_timestamp"].max().date()}')

## Step 2: Build Cohort Table via SQL

This replicates the logic in `sql/04_cohort_analysis.sql` in Python for the heatmap visualization.

In [ ]:
cohort_sql = '''
WITH first_purchase AS (
    SELECT
        c.customer_unique_id,
        MIN(strftime('%Y-%m', o.order_purchase_timestamp)) AS cohort_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_purchase_timestamp IS NOT NULL
      AND o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_unique_id
),
all_orders AS (
    SELECT
        c.customer_unique_id,
        fp.cohort_month,
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN first_purchase fp ON c.customer_unique_id = fp.customer_unique_id
    WHERE o.order_purchase_timestamp IS NOT NULL
      AND o.order_status NOT IN ('canceled', 'unavailable')
),
cohort_periods AS (
    SELECT
        cohort_month, order_month, customer_unique_id,
        (
            (CAST(substr(order_month,1,4) AS INTEGER)*12 + CAST(substr(order_month,6,2) AS INTEGER))
            - (CAST(substr(cohort_month,1,4) AS INTEGER)*12 + CAST(substr(cohort_month,6,2) AS INTEGER))
        ) AS period_number
    FROM all_orders
)
SELECT
    cohort_month,
    period_number,
    COUNT(DISTINCT customer_unique_id) AS customers
FROM cohort_periods
GROUP BY cohort_month, period_number
ORDER BY cohort_month, period_number
'''

cohort_raw = pd.read_sql_query(cohort_sql, engine)
print(f'Cohort records: {len(cohort_raw):,}')
print(f'Cohort months : {cohort_raw["cohort_month"].nunique()}')
cohort_raw.head(10)

In [ ]:
# Pivot to matrix: rows = cohort_month, columns = period_number
cohort_pivot = cohort_raw.pivot(
    index='cohort_month', columns='period_number', values='customers'
)

# Focus on 2017+ cohorts for complete data (2016 cohorts have partial months)
cohort_pivot = cohort_pivot[cohort_pivot.index >= '2017-01']

print(f'Cohort matrix shape: {cohort_pivot.shape}')
cohort_pivot.iloc[:5, :7]

## Step 3: Compute Retention Rates

Divide each cell by the cohort size (period 0) to get retention as a percentage.

In [ ]:
retention = build_retention_rates(cohort_pivot)
print('Retention rate matrix (first 5 cohorts, first 7 periods):')
retention.iloc[:5, :7].round(2)

## Step 4: Cohort Retention Heatmap

The heatmap rows are cohort months; columns are months since first purchase (0, 1, 2, …). The diagonal pattern is expected — cohorts with longer history have more filled columns.

In [ ]:
fig = plot_cohort_heatmap(retention, max_periods=12)
save_fig(fig, 'cohort_retention_heatmap.png')
plt.show()

## Step 5: Average Retention Rates at 1, 3, and 6 Months

In [ ]:
avg_ret = compute_avg_retention(retention)

print('Average Cohort Retention Rates:')
print(f'  1-month  retention: {avg_ret["m1"]:.2f}%')
print(f'  3-month  retention: {avg_ret["m3"]:.2f}%')
print(f'  6-month  retention: {avg_ret["m6"]:.2f}%')

# Save retention summary for Power BI notebook
import json
import os
reports_dir = project_root / 'reports'
reports_dir.mkdir(exist_ok=True)
existing = {}
kpi_path = reports_dir / 'kpi_summary.json'
if kpi_path.exists():
    existing = json.loads(kpi_path.read_text())
existing['cohort_retention'] = avg_ret
kpi_path.write_text(json.dumps(existing, indent=2))
print(f'\nRetention rates saved to {kpi_path}')

## Step 6: Save Cohort Map to Database

Stores each customer's cohort month in the database so notebook 05 can attach it to the Power BI export.

In [ ]:
cohort_map_sql = '''
SELECT
    c.customer_unique_id,
    MIN(strftime('%Y-%m', o.order_purchase_timestamp)) AS cohort_month
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_purchase_timestamp IS NOT NULL
  AND o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY c.customer_unique_id
'''

cohort_map = pd.read_sql_query(cohort_map_sql, engine)
cohort_map.to_sql('customer_cohorts', engine, if_exists='replace', index=False)
print(f'customer_cohorts table saved: {len(cohort_map):,} rows')

## Step 7: Retention Curve — Best vs Worst Cohorts

In [ ]:
# Plot top 3 and bottom 3 cohorts by 3-month retention
if 3 in retention.columns:
    m3 = retention[3].dropna().sort_values(ascending=False)
    best_cohorts  = m3.head(3).index.tolist()
    worst_cohorts = m3.tail(3).index.tolist()
    selected = best_cohorts + worst_cohorts

    periods = [c for c in range(0, 10) if c in retention.columns]
    fig, ax = plt.subplots(figsize=(11, 5))
    for cohort in selected:
        if cohort in retention.index:
            vals = retention.loc[cohort, periods].values
            label = f'{cohort} (3mo: {retention.loc[cohort, 3]:.1f}%)' if 3 in retention.columns else cohort
            ax.plot(periods, vals, marker='o', label=str(cohort))

    ax.set_title('Retention Curves — Selected Cohorts', fontsize=12)
    ax.set_xlabel('Months Since First Purchase')
    ax.set_ylabel('Retention Rate (%)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, 'cohort_retention_curves.png')
    plt.show()

## Summary & Business Interpretation

**Key finding:** Olist shows very low cohort retention — typically **under 10% of customers make a repeat purchase within 6 months**.

**Why this is expected:**
- Olist is a marketplace connecting independent sellers; customers often search for a specific product once
- There is no built-in loyalty program or subscription mechanism
- Many purchases are one-off (white goods, specific gifts, seasonal items)

**Business opportunity:**
This finding directly validates the need for:
1. A post-purchase email sequence encouraging cross-category discovery
2. A loyalty rewards program (points per purchase)
3. Category-based re-engagement campaigns targeting lapsed customers

Even a 2–3% improvement in 3-month retention across 90,000+ customers would add hundreds of additional orders per month.

**Next:** Run `04_rfm_segmentation.ipynb` to identify which customers are most worth targeting.